# TwoParticleControlledVSumBlockEncoding Call-Graph Visualization

This experiment builds a small `TwoParticleControlledVSumBlockEncoding` instance and visualizes its Qualtran call graph.

In [6]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from qualtran.drawing import show_call_graph, show_counts_sigma
from qualtran.resource_counting._call_graph import get_bloq_call_graph

if (Path.cwd() / 'src').exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / 'src').exists():
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from integrations.qualtran.block_encoding.two_particle_v_sum_block_encoding import (
    build_two_particle_controlled_v_sum_block_encoding,
)


In [7]:
# Small instance to keep decomposition and plotting lightweight.
D = 1
L = 2
R_c = 1
R_loc = 1
num_pairs = 2
entry_bitsize = 6

shape = (L,) * D + (L,) * D + (2 * R_c + 1,) * D + (2 * R_loc + 1,) * D
V_table = np.full(shape, 0.1, dtype=np.float64)

bloq = build_two_particle_controlled_v_sum_block_encoding(
    num_pairs=num_pairs,
    D=D,
    L=L,
    R_c=R_c,
    R_loc=R_loc,
    V_table=V_table,
    entry_bitsize=entry_bitsize,
)

print('Bloq:', type(bloq).__name__)
print('Signature:', bloq.signature)

Bloq: TwoParticleControlledVSumBlockEncoding
Signature: Signature((Register(name='ctrl', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='q', dtype=QBit(), _shape=(), side=<Side.THRU: 3>), Register(name='sel', dtype=BQUInt(bitsize=2, iteration_length=4), _shape=(), side=<Side.THRU: 3>), Register(name='m', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='l', dtype=BQUInt(bitsize=2, iteration_length=3), _shape=(1,), side=<Side.THRU: 3>), Register(name='r0', dtype=BQUInt(bitsize=1, iteration_length=2), _shape=(1,), side=<Side.THRU: 3>), Register(name='r1', dtype=BQUInt(bitsize=1, iteration_length=2), _shape=(1,), side=<Side.THRU: 3>), Register(name='r2', dtype=BQUInt(bitsize=1, iteration_length=2), _shape=(1,), side=<Side.THRU: 3>), Register(name='r3', dtype=BQUInt(bitsize=1, iteration_length=2), _shape=(1,), side=<Side.THRU: 3>)))


In [ ]:
# Build the recursive call graph, but keep QROM as a single unit (leaf).
# This plots the full structure while preventing QROM internals from expanding.

def _is_qrom_like(b):
    t = type(b)
    name = t.__name__.lower()
    mod = t.__module__.lower()
    return ('qrom' in name) or ('qram' in name) or ('data_loading' in mod and ('qrom' in mod or 'qram' in mod))


graph, sigma = get_bloq_call_graph(bloq, max_depth=6, keep=_is_qrom_like)

print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')

# Show top callees by total calls in the expanded graph.
top = sorted(sigma.items(), key=lambda kv: kv[1], reverse=True)[:20]
for callee, n in top:
    print(f'{type(callee).__name__:45s} {n}')


In [ ]:
# Visualize call graph with edge multiplicities.
fig, ax = plt.subplots(figsize=(15, 10))

pos = nx.spring_layout(graph, seed=7, k=0.8)
node_sizes = []
node_labels = {}
node_colors = []
for n in graph.nodes:
    calls = int(sigma.get(n, 1)) if str(sigma.get(n, 1)).isdigit() else 1
    node_sizes.append(320 + 8 * min(calls, 90))
    node_labels[n] = type(n).__name__
    node_colors.append('#fee8c8' if _is_qrom_like(n) else '#c6dbef')

nx.draw_networkx_nodes(graph, pos, node_size=node_sizes, node_color=node_colors, edgecolors='#1f4e79', ax=ax)
nx.draw_networkx_edges(graph, pos, arrows=True, arrowstyle='-|>', arrowsize=12, width=1.0, alpha=0.7, ax=ax)
nx.draw_networkx_labels(graph, pos, labels=node_labels, font_size=8, ax=ax)

edge_labels = {(u, v): d.get('n', '') for u, v, d in graph.edges(data=True)}
nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7, rotate=False, ax=ax)

ax.set_title('Call Graph: full expansion with QROM collapsed to unit nodes')
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Note:
# QROM nodes are included in the full graph but treated as leaf units via keep=_is_qrom_like.
# To expand QROM internals too, remove keep=_is_qrom_like in get_bloq_call_graph(...).
